This notebook is to upload the finetuned BERT models to Huggingface

In [5]:
#%pip install ipywidgets

## Log in to Hugging Face

In [6]:
from huggingface_hub import login
login()

## Add readable labels to the configs

The configs have no `id2label`, so the Hub widget and `pipeline` output show `LABEL_0`/`LABEL_1`. We write `NOT_<DV>` for 0 and `<DV>` for 1 into each `config.json` (e.g. `NOT_CONSERVATIVE`/`CONSERVATIVE`). This changes only the label names, not the weights or predictions. Class 1 is the coded dummy being 1: `scripts.train.py` renames each `*_DUMMY` column to `label`, and the class supports in `results.BERT_model_release_2026.csv` match the original test-set reports. The `models` dict defined here maps each local folder to its Hub name and is reused by the cells below.

In [7]:
import json
from pathlib import Path

models = {
    "CONSERVATIVE_DUMMY": "Conservative",
    "LIBERAL_DUMMY": "Liberal",
    "INCIVILITY_DUMMY": "Incivility",
    "INTERACTIVITY_DUMMY": "Interactivity",
    "RATIONALITY_DUMMY": "Rationality",
    "HAS_OPINION_DUMMY": "Has-Opinion",
    "HATELIST_FOCUSED_DUMMY": "Hatelist-Focused",
}

for folder in models:
    path = Path(folder) / "config.json"
    config = json.loads(path.read_text(encoding="utf-8"))
    label = folder.removesuffix("_DUMMY")
    config["id2label"] = {"0": f"NOT_{label}", "1": label}
    config["label2id"] = {f"NOT_{label}": 0, label: 1}
    path.write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
    print(folder, config["id2label"])

CONSERVATIVE_DUMMY {'0': 'NOT_CONSERVATIVE', '1': 'CONSERVATIVE'}
LIBERAL_DUMMY {'0': 'NOT_LIBERAL', '1': 'LIBERAL'}
INCIVILITY_DUMMY {'0': 'NOT_INCIVILITY', '1': 'INCIVILITY'}
INTERACTIVITY_DUMMY {'0': 'NOT_INTERACTIVITY', '1': 'INTERACTIVITY'}
RATIONALITY_DUMMY {'0': 'NOT_RATIONALITY', '1': 'RATIONALITY'}
HAS_OPINION_DUMMY {'0': 'NOT_HAS_OPINION', '1': 'HAS_OPINION'}
HATELIST_FOCUSED_DUMMY {'0': 'NOT_HATELIST_FOCUSED', '1': 'HATELIST_FOCUSED'}


All seven configs print the expected `{'0': 'NOT_<DV>', '1': '<DV>'}` pair, and `AutoConfig` reads them back with integer keys. The Hub widget and `pipeline` will now return e.g. `CONSERVATIVE` instead of `LABEL_1`. `scripts.evaluate.py` now looks predictions up in `model.config.label2id`, so it still works with these names. The cell can be re-run safely: it overwrites the same two keys.

## Write a model card per model

Each repo gets a `README.md` built from one template, so the seven cards share the same structure. Choices:
- Test scores come from `results.BERT_model_release_2026.csv`, the output of `scripts.evaluate.py` on the 773-comment test set. Per-class scores are reported next to accuracy, because the classes are imbalanced and accuracy alone overstates performance on the minority class.
- Script links are pinned to a commit, so they keep pointing at the code that produced these models even if the repository changes.
- The cards stress aggregate measurement over moderating individual comments, and state that these are retrained models whose scores differ from the paper.

In [8]:
import pandas as pd

COMMIT = "4803d34009c522b1015e602425cefbddee435a40"
REPO_NAME = "sjoerdstolwijk/Automatic-Social-Debate-Quality-Metrics"
REPO = f"https://github.com/{REPO_NAME}"
SCRIPTS = f"{REPO}/blob/{COMMIT}/BERT_model_release_2026"
CODEBOOK = "https://osf.io/cajqm/files/2g7ex"

definitions = {
    "CONSERVATIVE_DUMMY": "Whether the comment has a conservative/Republican ideological direction. Label 0 covers comments with no political opinion, a liberal direction, a neutral position (attacking or supporting both sides) or an unclear direction.",
    "LIBERAL_DUMMY": "Whether the comment has a liberal/Democrat ideological direction. Label 0 covers comments with no political opinion, a conservative direction, a neutral position (attacking or supporting both sides) or an unclear direction.",
    "INCIVILITY_DUMMY": "Whether the comment contains at least one explicit uncivil element: name-calling, vulgarity, questioning intelligence or competency, shouting, sarcasm, attacking a reputation, threatening individual rights, discrimination, or invoking violence.",
    "INTERACTIVITY_DUMMY": "Whether the substance of the comment refers back to a previous comment or claim of another commenter, showing that the commenter acknowledged other comments and that a real exchange of ideas could have taken place.",
    "RATIONALITY_DUMMY": "Whether the comment contains at least one of: (1) explicit reasoning or argumentation, such as elaborating on an opinion with \"because\"; (2) analysis of the background of the issue or background information; (3) external evidence, such as facts, figures or verifiable sources.",
    "HAS_OPINION_DUMMY": "Whether the comment expresses a position (either in favour/supporting or against) on a given societal or political issue.",
    "HATELIST_FOCUSED_DUMMY": "Whether the comment contains at least one of five uncivil elements: name-calling, vulgarity, threatening individual rights, discrimination, or invoking violence. It is a narrower subset of the nine incivility items. In this repository's analyses of the paper's measures (not in the paper itself), this measure is labelled *hate speech*. It is broader than most hate-speech definitions, because it also counts name-calling and vulgarity that are not aimed at a group.",
}

CITATION = """@article{stolwijk2026automated,
  title   = {Can we use automated approaches to measure the quality of online political discussion? How to (not) measure interactivity, diversity, rationality, and incivility in online comments to the news},
  author  = {Stolwijk, Sjoerd B. and Boukes, Mark and Yeung, Wang Ngai and Liao, Yufang and M{\\"u}nker, Simon and Kroon, Anne C. and Trilling, Damian},
  journal = {Communication Methods and Measures},
  year    = {2026},
  volume  = {20},
  number  = {1},
  pages   = {1--25},
  doi     = {10.1080/19312458.2025.2553300}
}"""

TEMPLATE = """---
license: cc-by-4.0
language: en
base_model: google-bert/bert-base-uncased
pipeline_tag: text-classification
tags:
- text-classification
- bert
- debate-quality
- deliberation
- user-comments
- computational-social-science
model-index:
- name: bert-base-uncased-CMM-2025-debate-quality-{dv}
  results:
  - task:
      type: text-classification
    dataset:
      name: CMM debate-quality comments (test split)
      type: custom
    metrics:
    - type: accuracy
      value: {acc:.3f}
    - type: f1
      name: macro F1
      value: {macro_f1:.3f}
---

# bert-base-uncased-CMM-2025-debate-quality-{dv}

Binary classifier (label 1 = `{label}`, label 0 = `NOT_{label}`) for one dimension of debate quality in online comments to the news. Retrained for the 2026 model release that accompanies Stolwijk et al. (2026).

**Definition:** {definition}

Full codebook: {codebook}

Training and evaluation code: the [`BERT_model_release_2026`]({scripts}) folder of the GitHub repository [{repo_name}]({repo}).

## Intended use

Measuring the prevalence of {dv} across large sets of English user comments on U.S. TV news, for research purposes.

**Out of scope:** moderating, ranking or sanctioning individual comments or users; languages other than English; platforms or topics far from the training data.

## Training data

- 3,862 manually coded user comments replying to TV news items posted on YouTube and Twitter (now X). The comments were collected in fall 2019 and replied to a wide variety of the most popular U.S. news shows at the time: nine regular news programs, five partisan news shows from the left and the right (MSNBC and Fox News), and seven satirical news programs.
- Split into 3,089 train and 773 test comments with [`scripts.split_data.py`]({scripts}/scripts.split_data.py).
- For GDPR and ethics reasons, the data is available for research purposes on request only.

## Training procedure

Fine-tuned from `google-bert/bert-base-uncased` with [`scripts.train.py`]({scripts}/scripts.train.py):
- learning rate 2e-5, 4 epochs, weight decay 0.01, batch size 16
- inputs truncated to 512 tokens; data shuffle seed 13
- no validation set or early stopping: the final checkpoint is released.

## Evaluation (test set, n = {n_test})

| Class | Precision | Recall | F1 | Support |
|---|---|---|---|---|
{rows}

Accuracy: {acc:.3f}

Scores come from [`scripts.evaluate.py`]({scripts}/scripts.evaluate.py). These are retrained models, so the scores differ from those reported in the paper.

## Usage

```python
from transformers import pipeline

clf = pipeline("text-classification", model="sjoerdstolwijk/bert-base-uncased-CMM-2025-debate-quality-{dv}")
clf("Your comment here")
```

## Code

[split_data]({scripts}/scripts.split_data.py) · [train]({scripts}/scripts.train.py) · [evaluate]({scripts}/scripts.evaluate.py) · [repository]({repo})

## Citation

```bibtex
{citation}
```

## AI notice

The model card and upload code were drafted with the help of Claude Opus 5.5 (`claude-opus-5-5`). The models themselves were trained without AI assistance.
"""

results = pd.read_csv("../results.BERT_model_release_2026.csv", index_col=0)

for folder, dv in models.items():
    label = folder.removesuffix("_DUMMY")
    r = results[results["Dimension"] == folder].set_index("Class")
    acc = r["Accuracy_overall"].iloc[0]
    n_test = int(r.loc["macro avg", "support"])
    rows = "\n".join(
        f"| {name} | {r.loc[c, 'Precision']:.3f} | {r.loc[c, 'Recall']:.3f} | {r.loc[c, 'F1-score']:.3f} | {int(r.loc[c, 'support'])} |"
        for c, name in [("0", f"0 (`NOT_{label}`)"), ("1", f"1 (`{label}`)"), ("macro avg", "Macro avg")]
    )
    card = TEMPLATE.format(dv=dv, label=label, definition=definitions[folder], codebook=CODEBOOK,
                           acc=acc, macro_f1=r.loc["macro avg", "F1-score"], n_test=n_test, rows=rows,
                           scripts=SCRIPTS, repo=REPO, repo_name=REPO_NAME, citation=CITATION)
    (Path(folder) / "README.md").write_text(card, encoding="utf-8")
    print(f"{dv:17} acc {acc:.3f}  class-1 F1 {r.loc['1', 'F1-score']:.3f}  macro F1 {r.loc['macro avg', 'F1-score']:.3f}")

Conservative      acc 0.853  class-1 F1 0.337  macro F1 0.627
Liberal           acc 0.799  class-1 F1 0.440  macro F1 0.659
Incivility        acc 0.761  class-1 F1 0.746  macro F1 0.760
Interactivity     acc 0.815  class-1 F1 0.671  macro F1 0.771
Rationality       acc 0.819  class-1 F1 0.489  macro F1 0.689
Has-Opinion       acc 0.787  class-1 F1 0.790  macro F1 0.786
Hatelist-Focused  acc 0.859  class-1 F1 0.643  macro F1 0.777


The cell writes seven `README.md` files and prints one score line per model:

| Model | Accuracy | Class-1 F1 | Macro F1 |
|---|---|---|---|
| Conservative | 0.853 | 0.337 | 0.627 |
| Liberal | 0.799 | 0.440 | 0.659 |
| Incivility | 0.761 | 0.746 | 0.760 |
| Interactivity | 0.815 | 0.671 | 0.771 |
| Rationality | 0.819 | 0.489 | 0.689 |
| Has-Opinion | 0.787 | 0.790 | 0.786 |
| Hatelist-Focused | 0.859 | 0.643 | 0.777 |

Accuracy and class-1 F1 tell different stories. Conservative has the second-highest accuracy but the lowest class-1 F1, because only 113 of 773 test comments are conservative. Liberal and Rationality show the same gap. These three models detect the presence of their concept poorly, which is why each card reports the per-class table and not only accuracy. Has-Opinion, Incivility and Interactivity are the most balanced.

The script links in the cards point to commit `4803d34` and only resolve once that commit is pushed to GitHub.

## Upload the seven models

Each model folder is uploaded to its own repository, `sjoerdstolwijk/bert-base-uncased-CMM-2025-debate-quality-{DV}`. Choices:
- `create_repo(..., exist_ok=True)` creates missing repos and leaves existing ones (e.g. the manually created Conservative repo) untouched.
- `upload_folder` adds or overwrites the files it sends, so the generated `README.md` replaces any existing card (including the one made by hand for Conservative). Files it doesn't send are left in place.
- `training_args.bin` is skipped: it is a pickle, not needed for inference, and flagged by the Hub's security scanner.
- New repos are created `private=True` so they can be checked before being made public.

In [9]:
from huggingface_hub import create_repo, upload_folder

for folder, dv in models.items():
    repo_id = f"sjoerdstolwijk/bert-base-uncased-CMM-2025-debate-quality-{dv}"
    create_repo(repo_id, repo_type="model", private=True, exist_ok=True)
    upload_folder(folder_path=Path(folder), repo_id=repo_id, repo_type="model",
                  ignore_patterns=["training_args.bin"])
    print(f"Uploaded {folder} -> https://huggingface.co/{repo_id}")

Uploaded CONSERVATIVE_DUMMY -> https://huggingface.co/sjoerdstolwijk/bert-base-uncased-CMM-2025-debate-quality-Conservative
Uploaded LIBERAL_DUMMY -> https://huggingface.co/sjoerdstolwijk/bert-base-uncased-CMM-2025-debate-quality-Liberal
Uploaded INCIVILITY_DUMMY -> https://huggingface.co/sjoerdstolwijk/bert-base-uncased-CMM-2025-debate-quality-Incivility
Uploaded INTERACTIVITY_DUMMY -> https://huggingface.co/sjoerdstolwijk/bert-base-uncased-CMM-2025-debate-quality-Interactivity
Uploaded RATIONALITY_DUMMY -> https://huggingface.co/sjoerdstolwijk/bert-base-uncased-CMM-2025-debate-quality-Rationality
Uploaded HAS_OPINION_DUMMY -> https://huggingface.co/sjoerdstolwijk/bert-base-uncased-CMM-2025-debate-quality-Has-Opinion
Uploaded HATELIST_FOCUSED_DUMMY -> https://huggingface.co/sjoerdstolwijk/bert-base-uncased-CMM-2025-debate-quality-Hatelist-Focused


## Check the result

The cell above prints one link per model; seven links means every upload succeeded. Open each repo to confirm it holds `config.json`, `model.safetensors` and the tokenizer files, check that the model card renders and the widget shows the readable labels, and switch the repos to public (Settings → Change visibility) once they are ready to be cited with the paper.